Chapter 2 dari buku Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow membahas proyek Machine Learning end-to-end, mulai dari pengambilan data hingga pembuatan pipeline. Di bawah ini adalah kode lengkap dari bagian awal proyek ini sampai preprocessing

💻 Kode Proyek End-to-End (California Housing Project)

1. Download dan Load Dataset

In [5]:
import os
import tarfile
import urllib.request

DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
HOUSING_PATH = os.path.join("datasets", "housing")
HOUSING_URL = DOWNLOAD_ROOT + "datasets/housing/housing.tgz"

def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    os.makedirs(housing_path, exist_ok=True)
    tgz_path = os.path.join(housing_path, "housing.tgz")
    urllib.request.urlretrieve(housing_url, tgz_path)
    housing_tgz = tarfile.open(tgz_path)
    housing_tgz.extractall(path=housing_path)
    housing_tgz.close()

fetch_housing_data()

In [6]:
import pandas as pd

def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = os.path.join(housing_path, "housing.csv")
    return pd.read_csv(csv_path)

housing = load_housing_data()
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


2. Split Data: Stratified Sampling Berdasarkan Income Category

In [7]:
import numpy as np
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

# Drop income_cat (tidak digunakan lagi)
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

3. Eksplorasi Data dan Kombinasi Atribut Baru

In [8]:
housing = strat_train_set.copy()

# Tambahkan atribut baru
housing["rooms_per_household"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["population_per_household"] = housing["population"] / housing["households"]

4. Pisahkan Fitur & Label (X & y)

In [9]:
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

5. Preprocessing Data Numerik & Kategorikal dengan Pipeline

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

# Tambahkan kolom kombinasi atribut
rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room=True):
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

# Buat pipeline numerik
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("attribs_adder", CombinedAttributesAdder()),
    ("std_scaler", StandardScaler()),
])

# Pisahkan kolom numerik & kategorikal
housing_num = housing.drop("ocean_proximity", axis=1)
num_attribs = list(housing_num)
cat_attribs = ["ocean_proximity"]

# Pipeline penuh
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs),
])

housing_prepared = full_pipeline.fit_transform(housing)

# 📘 Chapter 2: Proyek Machine Learning End-to-End

Chapter ini menunjukkan tahapan lengkap proyek Machine Learning dari awal hingga tahap persiapan model. Studi kasusnya adalah prediksi harga rumah di California menggunakan data sensus.

---

## 🧭 1. Gambaran Umum Proyek

- Dataset: **California Housing Dataset** (StatLib Repository)
- Target: Prediksi nilai median rumah (`median_house_value`)
- Tipe masalah: **Supervised Learning**, **Regression**
- Data dapat disimpan lokal dan dimuat menggunakan `pandas`

---

## 📊 2. Persiapan Dataset

### Langkah-langkah:
- **Ambil data** dari GitHub
- **Load** ke dalam `pandas.DataFrame`
- **Cek struktur data** menggunakan `.info()`, `.describe()`, dan `.head()`
- Deteksi kolom kategorikal (`ocean_proximity`) dan nilai hilang (`total_bedrooms`)

---

## 🧪 3. Membuat Set Uji

- Tujuan: hindari *data snooping bias*
- Gunakan **Stratified Sampling** berdasarkan kategori `median_income`
- Pisahkan data menjadi `train_set` dan `test_set`
- Gunakan `StratifiedShuffleSplit` dari `sklearn`

---

## 📍 4. Eksplorasi Data

- Visualisasi distribusi data dengan histogram (`.hist()`)
- Visualisasi geospasial: `longitude` vs `latitude` dengan scatter plot
- Visualisasi harga dengan ukuran dan warna (`c`, `s`, `cmap`)
- Lihat korelasi antar fitur dengan `.corr()`, `scatter_matrix()`

---

## ➕ 5. Kombinasi Fitur Baru

Tambahkan fitur kombinasi seperti:
- `rooms_per_household` = total_rooms / households
- `bedrooms_per_room` = total_bedrooms / total_rooms
- `population_per_household` = population / households

Tujuannya meningkatkan prediktivitas model.

---

## ⚙️ 6. Pisahkan Label dan Fitur

- Fitur: semua kolom kecuali `median_house_value`
- Label: `median_house_value`

```python
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()
